<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/tvd_mi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
# @title Dependencies

# Installations of third-party libraries
!pip install openai -q
!pip install anthropic -q
!pip install together -q

# First-party imports
import os
import json
import csv

# Third-party imports
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from together import Together
from google.colab import drive

import time
import random
from typing import List, Dict, Any, Optional, Tuple

In [ ]:
# @title Paths

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create PAPER_DIR for paper content
PAPER_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(PAPER_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {PAPER_DIR}.")

# Define and create LLM_DIR for llm based reviews
LLM_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(LLM_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {LLM_DIR}.")

# Define and create RES_DIR for the result file
RES_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(RES_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RES_DIR}.")

# Define and create RES_DIR for the log file
LOG_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {LOG_DIR}.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Thesis/.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/iclr/final.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/llm_reviews.


In [ ]:
# @title Definitions

MODEL = "gpt-4o-mini"
#MODEL = "claude-3-5-haiku-20241022"
#MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

TIMEOUT_SECONDS = 1200

OPENAI_KEY = "OPENAI_KEY_PLACEHOLDER"
ANTROPHIC_KEY = "ANTROPHIC_KEY_PLACEHOLDER"
TOGETHER_KEY = "TOGETHER_KEY_PLACEHOLDER"

In [ ]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(PAPER_DIR, "dataset_paper.parquet"))

# Select 'paper_id' and the target column 'parsed_text'
paper_content = dataset_paper[['paper_id', 'parsed_text']]

# Check paper_content
print(paper_content.head(3))

# Read llm reviews
dataset_llm = pd.read_csv(os.path.join(LLM_DIR, "llm_sim_results.csv"))

# Select 'paper_id' and 'review_text'
llm_reviews = dataset_llm[['document_id', 'review_text']] # change later to paper_id

# Check dataset_llm
print(llm_reviews.head(3))

      paper_id                                        parsed_text
0  vuD2xEtxZcj  INTRODUCTION Pruning Deep Neural Networks (DNN...
1  WoByU5W5te0  INTRODUCTION Recently, representing a 3D scene...
2  XZRmNjUMj0c  INTRODUCTION Neural Architecture Search (NAS) ...
   document_id                                        review_text
0  vuD2xEtxZcj  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...
1  WoByU5W5te0  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...
2  XZRmNjUMj0c  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...


In [ ]:
# @title Merge data

merged_df = pd.merge(paper_content, llm_reviews, left_on='paper_id', right_on='document_id', how='outer') # change later to paper_id
print(merged_df.head(3))
print(merged_df.shape)

     paper_id                                        parsed_text document_id  \
0  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   
1  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   
2  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   

                                         review_text  
0  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
1  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
2  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
(111664, 4)


# API

In [ ]:
# @title Define functions

# Cache for critic scores to avoid redundant API calls
cache = {}

def generate_tvd_mi_prompt(task_description: str, response_a: str, response_b: str) -> str:

    """Generate prompt for TVD-MI critic"""

    prompt = f"""You are evaluating whether two responses could be from agents working on the same task.

Task description: {task_description}

Response A: {response_a}

Response B: {response_b}

Do these responses show evidence of coming from the same task/source? Consider:
- Shared specific details, facts, or entities
- Similar topics or themes
- Overlapping information that would be unlikely if from different sources

Respond in two sentences or less. Put the information gain at the end in brackets in this format:

Rate the information gain:
- [[Significant Gain]]: Clear evidence they're from the same source (many shared specifics)
- [[Little Gain]]: Some shared elements but also differences
- [[No Gain]]: No evidence of shared source (could be from completely different tasks)"""

    return prompt

def interpret_tvd_mi_response(response: str) -> float:

    """Convert LLM response to numeric score"""

    response = response.strip().lower()

    if "[significant gain]" in response:
        return 1.0
    elif "[little gain]" in response:
        return 0.25
    elif "[no gain]" in response:
        return 0.0
    else:
        # Default to no gain if response is unclear
        print(f"Warning: Unclear response '{response}', defaulting to [[No Gain]]")
        return 0.0

class LLMClient:
    """
    Unified client handling Simulation, Caching, and Routing
    for OpenAI, Anthropic, and Together.
    """

    def __init__(self, simulate: bool = True, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.cache = {}
        self._openai = None
        self._anthropic = None
        self._together = None

    def _maybe_init_clients(self):
        if self.simulate:
            return

        # Note: Ensure OPENAI_KEY etc. are defined or use os.getenv
        if self._openai is None and "OPENAI_API_KEY" in globals():
            self._openai = OpenAI(api_key=OPENAI_API_KEY)
        if self._anthropic is None and "ANTHROPIC_API_KEY" in globals():
            self._anthropic = Anthropic(api_key=ANTHROPIC_API_KEY)
        if self._together is None and "TOGETHER_API_KEY" in globals():
            self._together = Together(api_key=TOGETHER_API_KEY)

    def call(
        self,
        model: str,
        prompt: str,
        temperature: float = 0.0,
        max_tokens: int = 500,
        max_retries: int = 5
    ) -> Tuple[str, Dict[str, Any]]:

        self._maybe_init_clients()

        # --- 1. SIMULATION PATH ---
        if self.simulate:
            raw = {
                "model": model,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "simulated_tvd_mi_score": (
                    f"This is a simulated TVD-MI score: Model='{model}', Temp={temperature}, Max Tokens='{max_tokens}'"
                )
            }
            structured = json.dumps(raw, indent=2)
            return structured, raw

        # --- 2. REAL API PATH ---
        for attempt in range(max_retries):
            try:
                response_text = ""
                provider = ""

                if "claude" in model.lower():
                    resp = self._anthropic.messages.create(
                        model=model,
                        max_tokens=max_tokens,
                        temperature=temperature,
                        messages=[{"role": "user", "content": prompt}]
                    )
                    response_text = resp.content[0].text
                    provider = "anthropic"

                elif "gpt" in model.lower():
                    resp = self._openai.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens
                    )
                    response_text = resp.choices[0].message.content
                    provider = "openai"

                else:
                    resp = self._together.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens
                    )
                    response_text = resp.choices[0].message.content
                    provider = "together"

                # ALIGNMENT STEP:
                # To match the simulation, we wrap the real response in the same dict structure
                raw = {
                    "model": model,
                    "temperature": temperature,
                    "max_tokens": max_tokens,
                    "provider": provider,
                    "response": response_text
                }

                # Create the indented JSON string
                structured = json.dumps(raw, indent=2)

                return structured, raw

            except Exception as e:
                wait_time = (2 ** attempt) + random.random()
                print(f"Error on attempt {attempt+1}: {e}. Retrying in {wait_time:.2f}s...")
                time.sleep(wait_time)

        error_raw = {"status": "error", "model": model}
        return json.dumps(error_raw, indent=2), error_raw


In [ ]:
# @title Run TVD-MI calculation

# Initialize LLMClient in simulation mode
# Set simulate=False and provide actual API keys (OPENAI_KEY, ANTHROPIC_KEY, TOGETHER_KEY)
# if you want to make real API calls.
llm_client = LLMClient(simulate=True)

task_description = "Evaluate if the given paper content and its review were generated by agents working on the same task. The task is to review the scientific paper provided."

results = []

# Iterate through each row of the merged_df
for index, row in merged_df.iterrows():
    paper_id = row['paper_id']
    response_a = row['parsed_text'] # Content of the paper
    response_b = row['review_text'] # LLM based review of the paper

    # Generate the TVD-MI prompt
    tvd_mi_prompt = generate_tvd_mi_prompt(task_description, response_a, response_b)

    # Call the LLMClient
    # Using the MODEL defined in Definitions cell, e.g., 'gpt-4o-mini'
    structured_response, raw_response = llm_client.call(
        model=MODEL,
        prompt=tvd_mi_prompt,
        temperature=0.0, # As used by Robertson et al.
        max_tokens=500 # As used by Robertson et al.
    )

    # Interpret the LLM's response to get the TVD-MI score
    # For simulated responses, 'simulated_tvd_mi_score' holds the entire string including the score part
    llm_critic_response_text = raw_response.get('response', raw_response.get('simulated_tvd_mi_score', ''))
    tvd_mi_score = interpret_tvd_mi_response(llm_critic_response_text)

    results.append({
        'paper_id': paper_id,
        'response_a': response_a,
        'response_b': response_b,
        'prompt': tvd_mi_prompt,
        'raw_response': raw_response,
        'structured_response': structured_response,
        'tvd_mi_score': tvd_mi_score
    })

# Convert results to a DataFrame
tvd_mi_results_df = pd.DataFrame(results)

# Check results
display(tvd_mi_results_df.head(5))

# Save results
tvd_mi_results_df.to_parquet(os.path.join(RES_DIR, "tvd_mi_scores.parquet"), index=False)

In [ ]:
# @title Check result